# D2 · Impurity importances reward noise: comparison with permutation

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Add to the Titanic two columns **with no information** (a continuous random number and a random ticket code with 100,000 values) and compare the impurity importances (`feature_importances_`, computed on the training set) with those by **permutation** on the test set: the first gives the two noise columns a large share, the second
brings them to zero.

## Dataset

`ds.titanic()` with two added columns: `random_number` (uniform) and `ticket_code` (random integers). Pipeline: median imputation for the numeric ones, mode + `OrdinalEncoder` for the categorical ones, `DecisionTreeClassifier(min_samples_leaf=5, random_state=0)`; stratified 70/30 split. `pipe` is already trained; `names` is the list of the column names in the order of the tree.

## Educational purpose

Know when **not** to trust `feature_importances_`. It is the best known bias of trees: a feature with many possible thresholds has more chances of finding a lucky one on the training set and the model uses it; the permutation importance on new data is not fooled.

## Task

1. Save `imp` (`pd.Series` with `feature_importances_`, index `names`) and `perm` (`pd.Series` with the mean of `permutation_importance(pipe, X_te, y_te, n_repeats=20, random_state=0, scoring="roc_auc").importances_mean`, index: the columns of `X_te` **reordered as `names`**).
2. Save `top5_impurity` and `top5_permutation`: the lists of the names of the 5 features with the largest importance according to each method.
3. Save `noise_share_impurity` (sum of the impurity importances of `random_number` and `ticket_code`) and `noise_share_permutation` (sum of their permutation values, **clipped at zero**, divided by the sum of all the positive values).
4. Also check `test_auc`: the test AUC of the pipeline. Is the tree, with the two useless columns, worse or better than a tree without them (`auc_without`)? Save `auc_without` (same tree without the two columns).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

df = ds.titanic()
X, y = df.drop(columns="survived").copy(), df["survived"]
r = np.random.default_rng(0)
X["random_number"] = r.uniform(size=len(X))
X["ticket_code"] = r.integers(0, 100000, len(X))
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare", "random_number", "ticket_code"]
CAT = ["sex", "port", "deck", "title"]
names = NUM + CAT
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)


def build(num, cat):
    prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), num), ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder()), cat)])
    return make_pipeline(prep, DecisionTreeClassifier(min_samples_leaf=5, random_state=0))


pipe = build(NUM, CAT).fit(X_tr, y_tr)

## Your work

In [ ]:
imp = None
perm = None

In [ ]:
top5_impurity = None
top5_permutation = None

In [ ]:
noise_share_impurity = None
noise_share_permutation = None

In [ ]:
test_auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
auc_without = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`perm_res = permutation_importance(pipe, X_te, y_te, n_repeats=20, random_state=0, scoring="roc_auc")`; its `importances_mean` are in the order of `X_te.columns`: `pd.Series(perm_res.importances_mean, index=X_te.columns).reindex(names)`. The impurity importances: `pd.Series(pipe[-1].feature_importances_, index=names)` (the `ColumnTransformer` puts the numeric ones first and then the categorical ones, as in `names`).'):
    check(list(imp.index) == names and close(imp.sum(), 1.0) and list(perm.index) == names, "the two series have the same names in the same order; the impurity importances sum to 1", "use X_te[names] or reindex(names) for the permutation")
    check("sex" in top5_impurity and "sex" in top5_permutation and top5_impurity[0] == "sex" and top5_permutation[0] == "sex", "`sex` is at the top of both", "the most important feature")
    check("random_number" in top5_impurity and "random_number" not in top5_permutation, "`random_number` is in the top 5 by impurity but not by permutation", "compare the two lists")
    check(noise_share_impurity > 0.15 and noise_share_permutation < 0.06, "the noise is worth over 15% of the impurity importance and less than 6% of the permutation one", "sum of the two useless features")
    check(auc_without is not None and auc_without > test_auc - 0.03, "without the useless columns the AUC does not get worse (and often improves)", "build(NUM without noise, CAT)")
done()

---
The commented solution is in `solutions/D_interpret/D2_impurity_importance_vs_permutation.ipynb`. Open it after you pass the check, or after you have really tried.